# 1. Create a database connection 🔌🏦

In [1]:
import pandas as pd
from sqlalchemy import create_engine, types
from sqlalchemy import text # to be able to pass string

In [2]:
# 1. Set up connection (update credentials and host!)
from dotenv import dotenv_values

config = dotenv_values() 

# define variables for the login
pg_user = config['POSTGRES_USER']  # align the key label with your .env file !
pg_host = config['POSTGRES_HOST']
pg_port = config['POSTGRES_PORT']
pg_db = config['POSTGRES_DB']
pg_schema = config['POSTGRES_SCHEMA']
pg_pass = config['POSTGRES_PASS']

url = f'postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}'

engine = create_engine(url, echo=False)



In [3]:
my_schema = "team_3"


mart_customers = pd.read_sql(f"SELECT * FROM {my_schema}.mart_customers;", engine)
mart_discount_coupon = pd.read_sql(f"SELECT * FROM {my_schema}.mart_discount_coupon;", engine)
mart_holidays_2019_us = pd.read_sql(f"SELECT * FROM {my_schema}.mart_holidays_2019_us;", engine)
mart_marketing_spend = pd.read_sql(f"SELECT * FROM {my_schema}.mart_marketing_spend;", engine)
mart_online_sales = pd.read_sql(f"SELECT * FROM {my_schema}.mart_online_sales;", engine)
mart_tax_amount = pd.read_sql(f"SELECT * FROM {my_schema}.mart_tax_amount;", engine)
mart_all_data = pd.read_sql(f"SELECT * FROM {my_schema}.mart_all_data;", engine)


1. CAC (Customer Acquisition Cost)

   CAC = Total Marketing Spend in Period / Number of New Customers Acquired in Same Period

In [4]:
print(mart_customers.columns)
print(mart_customers.head())

Index(['customer_id', 'gender', 'location', 'tenure_months'], dtype='object')
   customer_id gender    location  tenure_months
0        17850      M    Illinois             12
1        13047      M  California             43
2        12583      M    Illinois             33
3        13748      F  California             30
4        15100      M  California             49


In [5]:
print(mart_online_sales.columns)
print(mart_online_sales.head())

Index(['customer_id', 'transaction_id', 'transaction_date', 'produkt_sku',
       'product_category', 'quantity', 'avg_price', 'delivery_charges',
       'coupon_status', 'revenue', 'transaction_month',
       'transaction_month_str', 'transaction_day_of_month',
       'transaction_day_of_week'],
      dtype='object')
   customer_id  transaction_id transaction_date     produkt_sku  \
0        17850           16679       2019-01-01  GGOENEBJ079499   
1        17850           16680       2019-01-01  GGOENEBJ079499   
2        17850           16681       2019-01-01  GGOEGFKQ020399   
3        17850           16682       2019-01-01  GGOEGAAB010516   
4        17850           16682       2019-01-01  GGOEGBJL013999   

  product_category  quantity  avg_price  delivery_charges coupon_status  \
0         Nest-USA         1     153.71               6.5          Used   
1         Nest-USA         1     153.71               6.5          Used   
2           Office         1       2.05             

In [6]:
print(mart_marketing_spend.columns)
print(mart_marketing_spend.head())

Index(['date', 'offline_spend', 'online_spend'], dtype='object')
         date  offline_spend  online_spend
0  2019-01-01           4500       2424.50
1  2019-01-02           4500       3480.36
2  2019-01-03           4500       1576.38
3  2019-01-04           4500       2928.55
4  2019-01-05           4500       4055.30


In [7]:
print(mart_all_data.columns)
print(mart_marketing_spend.head())

Index(['transaction_date', 'transaction_id', 'customer_id', 'customer_gender',
       'customer_location', 'customer_tenure_months', 'produkt_sku',
       'product_category', 'quantity', 'avg_price', 'delivery_charges',
       'revenue', 'coupon_status', 'discount_pct', 'gst_onsale',
       'daily_offline', 'daily_online', 'transaction_month',
       'transaction_month_str'],
      dtype='object')
         date  offline_spend  online_spend
0  2019-01-01           4500       2424.50
1  2019-01-02           4500       3480.36
2  2019-01-03           4500       1576.38
3  2019-01-04           4500       2928.55
4  2019-01-05           4500       4055.30


In [8]:

# Make sure transaction_date is datetime
mart_online_sales['transaction_date'] = pd.to_datetime(mart_online_sales['transaction_date'])

# Extract year and month as strings (ensures match with spend table)
mart_online_sales['transaction_year'] = mart_online_sales['transaction_date'].dt.year.astype(str)
mart_online_sales['transaction_month_num'] = mart_online_sales['transaction_date'].dt.month.astype(str).str.zfill(2)
mart_online_sales['signup_month_str'] = mart_online_sales['transaction_year'] + '-' + mart_online_sales['transaction_month_num']

# Find first purchase date (and month) for each customer
first_orders = mart_online_sales.groupby('customer_id')['transaction_date'].min().reset_index()
first_orders['signup_year'] = first_orders['transaction_date'].dt.year.astype(str)
first_orders['signup_month_num'] = first_orders['transaction_date'].dt.month.astype(str).str.zfill(2)
first_orders['signup_month_str'] = first_orders['signup_year'] + '-' + first_orders['signup_month_num']


In [9]:
#count new customers per month
new_customers = first_orders.groupby('signup_month_str')['customer_id'].nunique().reset_index(name='new_customers')


In [10]:
#monthly marketing spend
mart_marketing_spend['date'] = pd.to_datetime(mart_marketing_spend['date'])
mart_marketing_spend['month_str'] = mart_marketing_spend['date'].dt.strftime('%Y-%m')
mart_marketing_spend['total_spend'] = (
    mart_marketing_spend['offline_spend'].fillna(0) +
    mart_marketing_spend['online_spend'].fillna(0)
)
monthly_spend = (
    mart_marketing_spend
    .groupby('month_str')['total_spend']
    .sum()
    .reset_index()
)

In [11]:
#merge and calculate CAC

# Ensure key is 'month_str' in both DataFrames
cac = pd.merge(
    new_customers,
    monthly_spend,
    left_on='signup_month_str',
    right_on='month_str',
    how='inner'
)
cac['CAC'] = cac['total_spend'] / cac['new_customers']

print(cac[['signup_month_str', 'new_customers', 'total_spend', 'CAC']])

   signup_month_str  new_customers  total_spend          CAC
0           2019-01            215    154928.95   720.599767
1           2019-02             96    137107.92  1428.207500
2           2019-03            177    122250.09   690.678475
3           2019-04            163    157026.83   963.354785
4           2019-05            112    118259.64  1055.889643
5           2019-06            137    134318.14   980.424380
6           2019-07             94    120217.85  1278.913298
7           2019-08            135    142904.15  1058.549259
8           2019-09             78    135514.54  1737.365897
9           2019-10             87    151224.65  1738.214368
10          2019-11             68    161144.96  2369.778824
11          2019-12            106    198648.75  1874.044811


2. ROAS (Return on Ad Spend) 

ROAS = Total Revenue / Total Marketing Spend

In [12]:
# Ensure transaction_date is datetime
mart_all_data['transaction_date'] = pd.to_datetime(mart_all_data['transaction_date'])

# Create the proper 'YYYY-MM' string for each transaction
mart_all_data['transaction_month_str'] = mart_all_data['transaction_date'].dt.strftime('%Y-%m')

monthly_revenue = mart_all_data.groupby('transaction_month_str')['revenue'].sum().reset_index(name='total_revenue')
print(monthly_revenue)


   transaction_month_str  total_revenue
0                2019-01      462866.90
1                2019-02      360036.40
2                2019-03      410408.03
3                2019-04      443100.16
4                2019-05      349159.59
5                2019-06      358594.96
6                2019-07      421362.00
7                2019-08      462309.94
8                2019-09      401553.82
9                2019-10      455643.16
10               2019-11      541254.55
11               2019-12      561140.18


In [13]:
mart_marketing_spend['date'] = pd.to_datetime(mart_marketing_spend['date'])
mart_marketing_spend['month_str'] = mart_marketing_spend['date'].dt.strftime('%Y-%m')

In [14]:
roas = pd.merge(
    monthly_revenue,
    monthly_spend,
    left_on='transaction_month_str',
    right_on='month_str',
    how='inner'
)
roas['ROAS'] = roas['total_revenue'] / roas['total_spend']
print(roas[['transaction_month_str', 'total_revenue', 'total_spend', 'ROAS']])


   transaction_month_str  total_revenue  total_spend      ROAS
0                2019-01      462866.90    154928.95  2.987608
1                2019-02      360036.40    137107.92  2.625934
2                2019-03      410408.03    122250.09  3.357118
3                2019-04      443100.16    157026.83  2.821812
4                2019-05      349159.59    118259.64  2.952483
5                2019-06      358594.96    134318.14  2.669743
6                2019-07      421362.00    120217.85  3.504987
7                2019-08      462309.94    142904.15  3.235105
8                2019-09      401553.82    135514.54  2.963179
9                2019-10      455643.16    151224.65  3.013022
10               2019-11      541254.55    161144.96  3.358805
11               2019-12      561140.18    198648.75  2.824786


In [15]:
roas.head()


,transaction_month_str,total_revenue,month_str,total_spend,ROAS
0,2019-01,462866.90,2019-01,154928.95,2.987608
1,2019-02,360036.40,2019-02,137107.92,2.625934
2,2019-03,410408.03,2019-03,122250.09,3.357118
3,2019-04,443100.16,2019-04,157026.83,2.821812
4,2019-05,349159.59,2019-05,118259.64,2.952483


3. Retention rate

In [16]:
# 1. Orders: customer_id, transaction_date
orders = mart_all_data[['customer_id', 'transaction_date']].drop_duplicates()
orders['order_month'] = pd.to_datetime(orders['transaction_date']).dt.to_period('M')  # 'YYYY-MM'

# 2. First purchase month per customer
first_orders = (
    orders.groupby('customer_id')['order_month']
    .min()
    .reset_index(name='signup_month')
)

# 3. Merge for cohort tracking
cohort_data = orders.merge(first_orders, on='customer_id', how='left')

# 4. Count unique customers per cohort and order month
cohort_pivot = (
    cohort_data
    .groupby(['signup_month', 'order_month'])['customer_id']
    .nunique()
    .reset_index()
)



In [17]:
# 5. Pivot to get cohort matrix
cohort_matrix = cohort_pivot.pivot(index='signup_month', columns='order_month', values='customer_id').fillna(0)



In [18]:
# 6. New customers per cohort
new_customers = cohort_matrix.iloc[:, 0]



In [19]:
# 7. Calculate retention rate (%)
retention_rate = (
    cohort_matrix.divide(new_customers.replace(0, pd.NA), axis=0) * 100
).fillna(0).replace([float('inf'), -float('inf')], 0)

# 8. (Optional) Filter to cohorts with >0 new customers
retention_rate = retention_rate.loc[new_customers > 0]

print(retention_rate)

order_month   2019-01   2019-02    2019-03    2019-04    2019-05    2019-06  \
signup_month                                                                  
2019-01         100.0  6.046512  11.162791  15.813953  10.697674  20.465116   

order_month    2019-07    2019-08    2019-09    2019-10   2019-11    2019-12  
signup_month                                                                  
2019-01       16.27907  21.860465  10.697674  13.023256  9.302326  15.813953  


C:\Users\Lukas\AppData\Local\Temp\ipykernel_27880\852977266.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0).replace([float('inf'), -float('inf')], 0)


4. AOV (AVerage order value)

AOV= Total Revenue/Number of Orders


In [20]:
monthly_orders = mart_all_data.groupby('transaction_month_str')['transaction_id'].nunique().reset_index(name='order_count')
monthly_revenue = mart_all_data.groupby('transaction_month_str')['revenue'].sum().reset_index(name='total_revenue')

monthly_aov = pd.merge(monthly_revenue, monthly_orders, on='transaction_month_str', how='inner')
monthly_aov['AOV'] = monthly_aov['total_revenue'] / monthly_aov['order_count']
print(monthly_aov[['transaction_month_str', 'AOV']])

   transaction_month_str         AOV
0                2019-01  220.203092
1                2019-02  216.368029
2                2019-03  206.131607
3                2019-04  244.401633
4                2019-05  171.661549
5                2019-06  184.842763
6                2019-07  202.577885
7                2019-08  191.511988
8                2019-09  207.843592
9                2019-10  214.420311
10               2019-11  237.184290
11               2019-12  209.068621


5. Repeat Purchase Rate

Repeat Purchase Rate= Number of Customers with >1 Order/ Total Number of Customers


In [21]:
repeat_customers = mart_all_data.groupby('customer_id')['transaction_id'].nunique().reset_index()
repeat_customers['is_repeat'] = repeat_customers['transaction_id'] > 1
repeat_rate = repeat_customers['is_repeat'].mean()
print(f"Repeat purchase rate: {repeat_rate:.2%}")

Repeat purchase rate: 91.49%


6. Churn rate

Churn Rate= Number of Customers with No Purchases for >6 Months/ Total Number of Customers

Approximation: 
Share of customers who haven’t purchased for at least 6 months, as a proxy for churn.

In [22]:
# Find last purchase date per customer
last_purchase = mart_all_data.groupby('customer_id')['transaction_date'].max().reset_index()
last_purchase['transaction_date'] = pd.to_datetime(last_purchase['transaction_date'])

reference_date = pd.to_datetime('2019-12-31')

# Calculate months since last purchase
last_purchase['months_since'] = ((reference_date - last_purchase['transaction_date']) / pd.Timedelta(days=30)).astype(int)

# Customers with >6 months inactivity are considered churned
churned = last_purchase[last_purchase['months_since'] > 6]

# Churn rate = churned customers / total unique customers
churn_rate = len(churned) / mart_all_data['customer_id'].nunique()

print(f"Churn rate (>6 months no purchase): {churn_rate:.2%}")

Churn rate (>6 months no purchase): 26.77%


7. LTV:CAC Ratio 

LTV = Total Revenue in period / Total Number of Customers acquired in period
CAC = Total Marketing Spend in period / Number of New Customers in period
LTV:CAC Ratio = LTV / CAC

In [23]:
ltv = mart_all_data.groupby('customer_id')['revenue'].sum().mean()  # Average revenue per customer

# Use your previously calculated CAC (from the 'cac' DataFrame)
avg_cac = cac['CAC'].mean()  # Average CAC, or use latest_cac = cac['CAC'].iloc[-1] for most recent

ltv_cac_ratio = ltv / avg_cac
print(f"LTV:CAC Ratio: {ltv_cac_ratio:.2f}")

LTV:CAC Ratio: 2.69


Benchmarks:

LTV:CAC > 1 → Your business is sustainable; each customer brings in more revenue than their acquisition cost.

LTV:CAC < 1 → You are overspending on customer acquisition.

8. ROAS online only

In [25]:
mart_all_data.head()

,transaction_date,transaction_id,customer_id,customer_gender,customer_location,customer_tenure_months,produkt_sku,product_category,quantity,avg_price,delivery_charges,revenue,coupon_status,discount_pct,gst_onsale,daily_offline,daily_online,transaction_month,transaction_month_str
0,2019-04-19,25990,15811,F,Illinois,27,GGOEWCKQ085457,Accessories,1,16.99,6.0,22.99,Clicked,10.0,0.1,4000,1754.92,4.0,2019-04
1,2019-04-07,25038,17999,F,New Jersey,30,GGOEGBPB081999,Accessories,1,49.99,6.5,56.49,Used,10.0,0.1,2500,2719.46,4.0,2019-04
2,2019-04-07,25038,17999,F,New Jersey,30,GGOEGBPB082099,Accessories,1,59.99,6.5,66.49,Clicked,10.0,0.1,2500,2719.46,4.0,2019-04
3,2019-04-28,26637,18116,F,Illinois,38,GGOEGBPB081999,Accessories,1,39.99,6.0,45.99,Clicked,10.0,0.1,3500,2019.73,4.0,2019-04
4,2019-08-02,34452,16889,F,Illinois,23,GGOEGBPB081999,Accessories,1,34.99,6.0,40.99,Used,20.0,0.1,1500,2155.96,8.0,2019-08


In [26]:
mart_all_data['daily_online'] = mart_all_data['daily_online'].fillna(0)
monthly_online_spend = mart_all_data.groupby('transaction_month_str')['daily_online'].sum().reset_index(name='online_spend')

In [27]:
mart_marketing_spend['online_spend'] = mart_marketing_spend['online_spend'].fillna(0)
monthly_online_spend = (
    mart_marketing_spend
    .groupby('month_str')['online_spend']
    .sum()
    .reset_index()
)


In [28]:
roas_online = pd.merge(
    monthly_revenue,
    monthly_online_spend,
    left_on='transaction_month_str',
    right_on='month_str',
    how='inner'
)
roas_online['roas_online'] = roas_online['total_revenue'] / roas_online['online_spend']

print(roas_online[['transaction_month_str', 'total_revenue', 'online_spend', 'roas_online']])


   transaction_month_str  total_revenue  online_spend  roas_online
0                2019-01      462866.90      58328.95     7.935457
1                2019-02      360036.40      55807.92     6.451350
2                2019-03      410408.03      48750.09     8.418611
3                2019-04      443100.16      61026.83     7.260744
4                2019-05      349159.59      52759.64     6.617930
5                2019-06      358594.96      53818.14     6.663087
6                2019-07      421362.00      52717.85     7.992777
7                2019-08      462309.94      57404.15     8.053598
8                2019-09      401553.82      52514.54     7.646526
9                2019-10      455643.16      57724.65     7.893390
10               2019-11      541254.55      68144.96     7.942694
11               2019-12      561140.18      76648.75     7.320931
